In [ ]:
%pip install boto3 --upgrade --break-system-packages ;

# Environment Setup for Mcity Data Engine (AWS Cloud)

To initialize the Mcity Data Engine in the AWS cloud environment, please follow these instructions for setting up your AWS credentials, roles, and required policies.

---

## 1. IAM Role and User Configuration

**a. Create IAM Role**

Create a new AWS IAM role named `mcity-data-engine-agent-cf-role` with these managed permission policies:

Trusted Policy:
```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "cloudformation.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
```

Policy:

- `AmazonEC2FullAccess`
- `AmazonSSMFullAccess`
- `AWSCloudFormationFullAccess`

**b. Create IAM User**

Create an IAM user called `mcity-data-engine-user` and attach the following custom policy:

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "VisualEditor0",
            "Effect": "Allow",
            "Action": [
                "cloudformation:DescribeStackEvents",
                "cloudformation:CreateStack",
                "cloudformation:GetTemplate",
                "ssm:*",
                "cloudformation:DeleteStack",
                "ec2:*",
                "ec2:DescribeKeyPairs",
                "cloudformation:DescribeStacks"
            ],
            "Resource": "*"
        },
        {
            "Sid": "VisualEditor1",
            "Effect": "Allow",
            "Action": [
                "iam:GetRole",
                "iam:PassRole"
            ],
            "Resource": "arn:aws:iam::<ACCOUNT-ID>:role/mcity-data-engine-agent-cf-role"
        }
    ]
}


In [ ]:
import getpass
import boto3

# Securely prompt for credentials (input is obscured)
AWS_ACCESS_KEY = getpass.getpass("Enter AWS Access Key ID: ")

In [ ]:
AWS_SECRET_KEY = getpass.getpass("Enter AWS Secret Access Key: ")

In [ ]:
REGION='us-east-1'
OPENAIAPIKEY = getpass.getpass("Enter Open API Key: ")

In [ ]:
HFTOKEN =  getpass.getpass("Enter huggingface API Key: ")

In [ ]:
cf_client = boto3.client(
    'cloudformation',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)


ec2_client = boto3.client(
    'ec2',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

ssm_client = boto3.client(
    'ssm',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

In [ ]:
with open('deploy-agent-AWS-AMI.yml', 'r') as f:
    template_body = f.read()

response = cf_client.create_stack(
    StackName='mcity-data-engine-agent',
    TemplateBody=template_body,
    RoleARN='arn:aws:iam::1234567800000:role/mcity-data-engine-agent-cf-role',
    Parameters=[
        #{'ParameterKey': 'RepoURL', 'ParameterValue': 'github.com'},
        {'ParameterKey': 'KeyPairName', 'ParameterValue': 'mcity-data-engine-key'},
        {'ParameterKey': 'OPENAIAPIKEY', 'ParameterValue': OPENAIAPIKEY},
        {'ParameterKey': 'HFTOKEN', 'ParameterValue': HFTOKEN},
        {'ParameterKey': 'InstanceType', 'ParameterValue': 'r5.large'}
    ],
    Capabilities=['CAPABILITY_NAMED_IAM']
)



In [ ]:
waiter = cf_client.get_waiter('stack_create_complete')
waiter.wait(StackName='mcity-data-engine-agent')

key_pairs = ec2_client.describe_key_pairs(
    Filters=[{'Name': 'key-name', 'Values': ['mcity-data-engine-key']}]
)
key_pair_id = key_pairs['KeyPairs'][0]['KeyPairId']
parameter_name = f"/ec2/keypair/{key_pair_id}"
param_response = ssm_client.get_parameter(
    Name=parameter_name,
    WithDecryption=True
)

private_key = param_response['Parameter']['Value']

key_filename = "mcity-data-engine-key.pem"
with open(key_filename, "w") as f:
    f.write(private_key)
import os
os.chmod(key_filename, 0o600)    

In [ ]:
stack_info = cf_client.describe_stacks(StackName='mcity-data-engine-agent')
outputs = stack_info['Stacks'][0].get('Outputs', [])
public_ip = next(o['OutputValue'] for o in outputs if o['OutputKey'] == 'PublicIP')
print(f"Open url in chrome http://{public_ip}:5225")

In [ ]:
!ssh -i "mcity-data-engine-key.pem" -f -N -o StrictHostKeyChecking=no -L 5151:localhost:5151 ubuntu@{public_ip} 
print(f"Open url in chrome http://localhost:5151")

In [ ]:
 cf_client.delete_stack(
        StackName='mcity-data-engine-agent',
        # DeletionMode='FORCE_DELETE_STACK'
    )